# UCI heart disease analysis

**Executive summary:**

This notebook develops a predictive model for the presence of heart disease based on clinical and exercise-induced features from the processed Cleveland dataset. After cleaning and one-hot encoding categorical variables, I engineered new biologically motivated features (e.g. cholesterol per age, and exercise response indicators). I applied scaling and logistic regression with regularization to model the data based on results compared to tree-based models.

My final logistic regression model achieved a cross-validated F1 score of 0.80 and a test accuracy of 81%, with balanced performance across classes and strong generalization. This is right on par with previous efforts to model this dataset. Visualizations confirmed meaningful patterns between features and the disease outcome, and classification metrics indicated the model avoids overfitting.

Subsequent sections walk through data exploration, feature construction, model selection, and evaluation in detail.
 
**Dataset overview:**

Data found here: https://archive.ics.uci.edu/dataset/45/heart+disease 

The dataset has a total of 76 features but all published results only use 14 of them, and that is what we are provided for analysis here. Each section in this notebook will have key explanations and observations from that step of the analysis, and there will be a summary/conclusions section at the end.

**Assumptions/next steps:**

Assumptions/next steps given more time:
* We are just using Cleveland data for right now due to time constraints
    * If given more time, we could also use the data from Switzerland, Hungary, and Long Beach, but these datasets are less well-documented and would require more time for cleaning
* We are ignoring the cost folder for now due to the goal of the analysis
    * We are being asked to predict heart disease
    * If given more time, we could use the cost folder to do some analyses on cost-sensitive testing (e.g. minimizing cost while maxmizing model accuracy)
* The "num" feature in the processed Cleveland data (our target) is loaded with values 0-4 but we are binarizing the target
    * Any sample with a 0 num value is given a value of 0 in the binary target
    * All other values have a value of 1
    * This is presence/absence of disease
    * This is how most analyses have treated this variable so it makes sense to do so here

In [ ]:
import re
import pandas as pd
import numpy as np
import plotly.express as px
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.load import load_processed_cleveland
from src.preprocess import one_hot_encode, add_engineered_features
from src.utils import plot_confusion_matrix

## Load, clean, and explore data

In the processed data, several values like slope and restecg are categorical because they have been broken down into bins. For example, slope can be upsloping, flat, or downsloping. These categorical variables will need to be dummy encoded if they have more than two classes.

### Load, binarize target, drop missing values

In [ ]:
df = load_processed_cleveland('data/Module 2 heart+disease/processed.cleveland.data')
df.shape

In [ ]:
df.head()

In [ ]:
df.describe()

### Add engineered features

I engineered additional features based on domain knowledge of cardiovascular risk factors and exercise physiology that I got from a mix of Google and literature searching, as well as training on interaction effects from my graduate school days. While the original dataset includes standard clinical variables, many of these are most informative when considered in combination. For example, features like heart rate reserve and age-adjusted ST depression (age × oldpeak) help capture physiological responses to stress that are not fully expressed by any single variable. Engineering these features allowed me to provide the model with more biologically relevant signals, improve class separation in visualizations, and ultimately enhance predictive performance. See below for a description of each engineered feature:

chol_per_age (chol / age)
* Normalizes cholesterol by age to better capture long-term lipid exposure, which contributes to atherosclerosis and coronary artery disease.

bp_hr_ratio (trestbps / thalach)
* Relates resting blood pressure to cardiac output; a high ratio may indicate vascular resistance or inadequate heart response to stress.

age_oldpeak_interaction (age × oldpeak)
* Combines age-related risk with ECG response to stress; older individuals with significant ST depression often have advanced or silent ischemia.

abnormal_exercise_response (oldpeak > 2 and thalach < 120)
* Flags patients who struggle to raise heart rate and show ST depression, a classic sign of exercise-induced myocardial ischemia.

is_high_chol, is_hypertensive
* Flags established cardiovascular risk factors; both are strongly linked to coronary artery disease through endothelial damage and plaque formation.

In [ ]:
df = add_engineered_features(df)
df.shape

### Visualizations

Data histograms can let us see things like outliers and decide how to scale our features. The box plots show us which features might discriminate between our positive and negative classes.

Observations:
* We only drop 6 rows due to missing data
* Classes are pretty evenly balanced
* Most continuous features are normally distributed except for oldpeak, but some are skewed or have outliers
    * Consider using robustscaler
* Based on box plots, heart disease patients appear to be older, and have lower thalach values and higher oldpeak values than non-heart disease patients
    * Aging raises your risk of heart disease
    * Heart disease patients have lower max heart rates during stress tests
* Heart patients have ST depression from lack of oxygen to heart tissue during exercise
* Older people already have reduced cardiovascular reserve, so that + ST depression = strong indicator of heart disease
* bp_hr_ratio higher in heart disease, likely due to elevated BP and poor exercise tolerance

In [ ]:
sns.countplot(x="target", data=df)
plt.title("Distribution of Heart Disease Diagnosis")

In [ ]:
categorical_features = ["cp", "restecg", "slope", "thal", "ca"]
cont_feats = ["age", "trestbps", "chol", "thalach", "oldpeak", "chol_per_age", "bp_hr_ratio", "age_oldpeak_interaction"]

In [ ]:
for col in cont_feats:
    sns.histplot(data=df, x=col, hue="target", kde=True, element="step", stat="density")
    plt.title(f"{col} distribution by diagnosis")
    plt.show()

In [ ]:
for col in cont_feats:
    sns.boxplot(x="target", y=col, data=df)
    plt.title(f"{col} by Heart Disease Diagnosis")
    plt.show()

### One hot encode categorical features

Models need numerical features, so this step is necessary even if the categorical features come as numbers (e.g. if 0 represents a flat slope, 1 an upward slope, and 2 a downward slope, we still need to onehot encode the different categories). 

In [ ]:
df_encoded = one_hot_encode(df, categorical_features)

In [ ]:
cols_to_convert = [col for col in df_encoded.columns if col not in cont_feats + ["target"]]
df_encoded[cols_to_convert] = df_encoded[cols_to_convert].astype(int)

### Split data

Split into train and test sets. We do this to reserve a holdout set for model evaluation. This must not be involved in model selection at all, including feature selection and model training/cross validation in order to avoid leakage. This gives us an idea of how well the model will generalize to unseen data.

Must do this before scaling!

In [ ]:
X = df_encoded.drop(['target'], axis=1)
y = df_encoded[['target']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1738, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

### Scale continous features

Scaling is necessary for some models like SVM because features that are on higher scales might get more weight in the model just becasue their values are bigger, and not because they're actually more discriminative between the target classes. For example, a feature that ranges from 100-200 might have more effect in an SVM than a feature that ranges from 0-2. Scaling addresses this issue.

Some features like cholesterol might have meaningful outliers (e.g. very bad cholesterol might correspond with bad heart disease), so I will use RobustScaler.

In [ ]:
scaler = RobustScaler()
X_train[cont_feats] = scaler.fit_transform(X_train[cont_feats])
X_test[cont_feats] = scaler.transform(X_test[cont_feats])
X_train.shape, X_test.shape

In [ ]:
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

### Explore correlations

See which features are correlated to one another and see if we can drop any due to high correlation. This is necessary if we want to compare SVM to RF to XGBoost since SVMs are susceptible to correlated features. This is because these features are redundant, so SVMs put more emphasis on them, but ensemble methods are more robust to this since they build trees separately with different feature subsets.

We can use Pearson correlation for this because it is safe for:
* Continuous vs. Continuous
* Binary vs Binary (equivalent to Phi coefficient)
* Continuous vs Binary (equivalent to Point-Biserial, a subtype of Pearson)

Observations:
* Nothing has an absolute correlation coefficient above 0.9 so we can keep everything

In [ ]:
corrmat = X_train.corr()
fig = px.imshow(corrmat, 
                labels=dict(x="Features", y="Features", color="Correlation"),
                x=corrmat.columns,
                y=corrmat.index,
                color_continuous_scale=px.colors.diverging.RdBu,
                range_color=[-1, 1])
fig.update_layout(title_text='Correlation Matrix', width=1000, height=1000)
fig.show()

### Correlation with target

Which features might be most predictive?

Observations:
* One of our onehot-encoded cp features is the most positively correlated feature
* thalach is the most negatively correlated
* There is only one feature that looks like it might be noise (almost no correlation), so we don't really need to do feature selection

In [ ]:
correlations = X_train.corrwith(y_train.target)
correlations = correlations.sort_values(ascending=False)
correlations = pd.DataFrame(correlations, columns=['correlation']).reset_index().rename(columns={'index':'feature'})

In [ ]:
fig = px.bar(
    correlations,
    x="correlation",
    y="feature",
    orientation="h",
    title="Feature–Target Correlation (Target = 1 = heart disease)",
    labels={"correlation": "Correlation with Heart Disease", "feature": "Feature"},
    color="correlation",
    color_continuous_scale="RdBu",
    range_color=[-1, 1],
    width=800,
    height=700
)

fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

### Feature selection

There are not very many features, all the models I want to try (XGBoost, SVM, RF, logistic regression) are all robust to this many features, and most contain at least some information. Let's keep em all!

So, we are including all original and engineered features in the model.

## Train model

Logistic regression ended up giving best results. This is a good choice for mixed datatypes like we have (binary and continuous) because it learns a separate coefficient for each feature. It's also a pretty simple and interpretable model which is always nice to have when clearing things with the FDA (boosts explainability). I use 5 fold cross validation to prevent overfitting and grid search so that I can select optimal hyperparameters.

Results:
* CV F1 score of 0.8 and accuracy of 0.89
* Holdout accuracy of 0.81 puts us at a very respectable score when compared with the results graph on the dataset's website
    * Also tells us we aren't overfitting - I'd worry if we saw something like high precision but low recall on test set or a bigger gap between train and test accuracy
    * Though it would be nice to bring up test set accuracy a bit more to close the gap if I had more time
        * Try a more thorough grid search with ensemble and SVM models
        * Be more intentional with feature selection - try eliminating any features with poor discrimination
* Plotting ROC/AUC might be useful with more time, but the classification report summarizes all this info

In [ ]:
logreg = LogisticRegression(solver='liblinear', max_iter=1000, random_state=1738)
param_grid = {
    'penalty': ['l1', 'l2'],
    'C': [0.01, 0.1, 1, 10, 100],  # Inverse of regularization strength
}
grid_search_lr = GridSearchCV(
    estimator=logreg,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=2
)
grid_search_lr.fit(X_train, y_train.target.values.ravel())
print("Best parameters:", grid_search_lr.best_params_)
print("Best CV F1 score:", grid_search_lr.best_score_)

In [ ]:
y_pred_lr = grid_search_lr.best_estimator_.predict(X_test)
print(classification_report(y_test["target"], y_pred_lr))

In [ ]:
plot_confusion_matrix(y_test["target"], y_pred_lr, title="Logistic Regression Confusion Matrix", class_names=["No disease", "Disease"])

## Conclusions/summary

I trained a logistic regression model to predict heart disease using a combination of clinical features and biologically informed engineered variables from the processed Cleveland dataset. After preprocessing steps including one-hot encoding and robust scaling of continuous variables, I performed hyperparameter tuning with cross-validation. The best model, which used L2 regularization with a regularization strength of C=10, achieved a cross-validated F1 score of 0.80. On the holdout test set, the model reached an accuracy of 81%, with balanced precision and recall across both classes. The classification report indicated strong performance in identifying heart disease cases, with a slight tradeoff in recall that helped preserve precision. Overall, the logistic regression model showed reliable generalization and interpretable decision boundaries, making it a strong baseline for clinical risk prediction.

In the future, it might make sense to tweak the decision threshold to favor recall (making the model better at detecting disease) since that migth be more clinically important than false positives. Recall that logistic regression decides a positive case if predict_proba > 0.5, so we could consider lowering this, but for the sake of time I haven't played around with that yet.